# [SOLUCIONARIO] Laboratorio 04 — Fundamentos de Redes Neuronales Artificiales
### Tecnologías Emergentes (ISO46B) — Semana 4 — Material del docente

Este notebook incluye el código completo y, adicionalmente, **celdas de verificación automática** (marcadas con ✅) que comprueban que cada paso produjo el resultado esperado (forma de los arreglos, rangos de precisión razonables, existencia de archivos, etc.). Úsalo como referencia al revisar las entregas o al proyectar la solución en clase.


## Parte 1 — Básico: Preparación del dataset

### Paso 1.1 — Cargar y explorar el dataset

In [ ]:
import pandas as pd

df = pd.read_csv("pronostico_huancayo.csv")

print("Dimensiones del dataset:", df.shape)
print("\nColumnas disponibles:")
print(df.columns.tolist())
df.head()

In [ ]:
# ✅ Verificación automática
assert df.shape[0] > 0, "El dataset está vacío"
assert {"temp_max", "temp_min", "precipitacion"}.issubset(df.columns), \
    "Faltan columnas esperadas: temp_max, temp_min, precipitacion"
print("✅ Verificación 1.1 OK: dataset cargado con las columnas esperadas.")

### Paso 1.2 — Construir la variable objetivo y las características

In [ ]:
def calcular_dia_lluvioso(precipitacion):
    """Devuelve 1 si hubo precipitación registrada, 0 en caso contrario."""
    return int(precipitacion > 0)


df["dia_lluvioso"] = df["precipitacion"].apply(calcular_dia_lluvioso)
df["amplitud_termica"] = df["temp_max"] - df["temp_min"]

columnas_features = ["temp_max", "temp_min", "amplitud_termica"]
X = df[columnas_features].copy()
y = df["dia_lluvioso"].copy()

print("Distribución de la variable objetivo:")
print(y.value_counts())

In [ ]:
# ✅ Verificación automática
assert calcular_dia_lluvioso(2.5) == 1
assert calcular_dia_lluvioso(0.0) == 0
assert set(y.unique()).issubset({0, 1}), "La variable objetivo debe ser binaria (0/1)"
assert "amplitud_termica" in X.columns
print("✅ Verificación 1.2 OK: variable objetivo binaria y amplitud_termica correctamente calculadas.")

### Paso 1.3 — Limpieza, escalado y partición train/test

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

datos_completos = pd.concat([X, y], axis=1).dropna()
X = datos_completos[columnas_features]
y = datos_completos["dia_lluvioso"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

escalador = StandardScaler()
X_train_esc = escalador.fit_transform(X_train)
X_test_esc = escalador.transform(X_test)

print("Tamaño de entrenamiento:", X_train_esc.shape)
print("Tamaño de prueba:", X_test_esc.shape)

In [ ]:
# ✅ Verificación automática
assert X_train_esc.shape[1] == 3, "Se esperan 3 características"
assert abs(len(X_test_esc) / len(X) - 0.2) < 0.05, "La partición test no está cerca del 20%"
assert abs(X_train_esc.mean()) < 0.5, "El escalado no parece centrado en 0 (revisar StandardScaler)"
print("✅ Verificación 1.3 OK: partición y escalado dentro de los rangos esperados.")

## Parte 2 — Intermedio: Primer MLP con Keras (datos sintéticos)

### Paso 2.1 — Generar datos sintéticos

In [ ]:
from sklearn.datasets import make_classification

X_sint, y_sint = make_classification(
    n_samples=500, n_features=4, n_informative=3, n_redundant=0, random_state=42
)

X_sint_train, X_sint_test, y_sint_train, y_sint_test = train_test_split(
    X_sint, y_sint, test_size=0.2, random_state=42
)

escalador_sint = StandardScaler()
X_sint_train = escalador_sint.fit_transform(X_sint_train)
X_sint_test = escalador_sint.transform(X_sint_test)

print("X_sint_train:", X_sint_train.shape, " X_sint_test:", X_sint_test.shape)

### Paso 2.2 — Definir la arquitectura del MLP

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

modelo_sintetico = keras.Sequential([
    layers.Input(shape=(4,)),
    layers.Dense(8, activation="relu"),
    layers.Dense(4, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

modelo_sintetico.summary()

In [ ]:
# ✅ Verificación automática
assert len(modelo_sintetico.layers) == 3, "Se esperan 3 capas Dense"
assert modelo_sintetico.layers[-1].units == 1, "La capa de salida debe tener 1 neurona"
assert modelo_sintetico.layers[-1].activation.__name__ == "sigmoid", \
    "La capa de salida debe usar activación sigmoid para clasificación binaria"
print("✅ Verificación 2.2 OK: arquitectura correcta (3 capas, salida sigmoid).")

### Paso 2.3 — Compilar el modelo

In [ ]:
modelo_sintetico.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
print("✅ Modelo compilado.")

### Paso 2.4 — Entrenar y graficar

In [ ]:
import matplotlib.pyplot as plt

historial = modelo_sintetico.fit(
    X_sint_train, y_sint_train,
    epochs=30, batch_size=16, validation_split=0.2, verbose=0,
)

fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
ejes[0].plot(historial.history["loss"], label="entrenamiento")
ejes[0].plot(historial.history["val_loss"], label="validación")
ejes[0].set_title("Pérdida (loss)")
ejes[0].legend()

ejes[1].plot(historial.history["accuracy"], label="entrenamiento")
ejes[1].plot(historial.history["val_accuracy"], label="validación")
ejes[1].set_title("Precisión (accuracy)")
ejes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# ✅ Verificación automática
assert len(historial.history["loss"]) == 30, "Se esperan 30 épocas de entrenamiento"
assert historial.history["loss"][-1] < historial.history["loss"][0], \
    "La pérdida final debería ser menor que la inicial (el modelo debe aprender)"
print("✅ Verificación 2.4 OK: el entrenamiento redujo la pérdida a lo largo de las épocas.")

### Paso 2.5 — Evaluar en el conjunto de prueba

In [ ]:
perdida_test, precision_test = modelo_sintetico.evaluate(X_sint_test, y_sint_test, verbose=0)
print(f"Pérdida en test: {perdida_test:.4f}")
print(f"Precisión en test: {precision_test:.4f}")

In [ ]:
# ✅ Verificación automática
assert 0.0 <= precision_test <= 1.0
assert precision_test > 0.6, \
    f"Precisión inusualmente baja ({precision_test:.2f}) para datos sintéticos; revisar arquitectura/entrenamiento"
print("✅ Verificación 2.5 OK: precisión en test dentro de un rango razonable (>0.60).")

## Parte 3 — Avanzado: Clasificador de lluvia (datos reales)

### Paso 3.1 — Modelo baseline: regresión logística

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

baseline = LogisticRegression()
baseline.fit(X_train_esc, y_train)
pred_baseline = baseline.predict(X_test_esc)
precision_baseline = accuracy_score(y_test, pred_baseline)
print(f"Precisión del baseline: {precision_baseline:.4f}")

### Paso 3.2 — Construir, compilar y entrenar el MLP sobre el dataset real

In [ ]:
modelo_lluvia = keras.Sequential([
    layers.Input(shape=(X_train_esc.shape[1],)),
    layers.Dense(8, activation="relu"),
    layers.Dense(4, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

modelo_lluvia.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

historial_lluvia = modelo_lluvia.fit(
    X_train_esc, y_train,
    epochs=40, batch_size=8, validation_split=0.2, verbose=0,
)

perdida_lluvia, precision_lluvia = modelo_lluvia.evaluate(X_test_esc, y_test, verbose=0)
print(f"Precisión del MLP en test: {precision_lluvia:.4f}")
print(f"Precisión del baseline en test: {precision_baseline:.4f}")

In [ ]:
# ✅ Verificación automática
assert 0.0 <= precision_lluvia <= 1.0
# No exigimos que el MLP supere al baseline: con un dataset pequeño es normal que no lo haga.
# Verificamos únicamente que el flujo completo produjo un resultado válido y comparable.
print("✅ Verificación 3.2 OK: MLP entrenado y evaluado; comparación con baseline disponible.")
print(f"   Diferencia MLP - baseline: {precision_lluvia - precision_baseline:+.4f}")

**Respuesta de referencia a la pregunta de reflexión:** con un dataset de menos de 100 registros y solo 3 características, es esperable que el MLP no supere claramente a la regresión logística —incluso puede rendir igual o algo peor—. Esto no indica un error de implementación: ilustra que la ventaja de una red neuronal sobre modelos más simples se manifiesta principalmente con MÁS DATOS y relaciones más complejas/no lineales entre variables, condiciones que este dataset reducido no cumple plenamente.

### Paso 3.3 — Guardar el modelo

In [ ]:
modelo_lluvia.save("modelo_lluvia.keras")
print("Modelo guardado en modelo_lluvia.keras")

In [ ]:
# ✅ Verificación automática
import os
assert os.path.exists("modelo_lluvia.keras"), "No se encontró el archivo modelo_lluvia.keras"
print("✅ Verificación 3.3 OK: archivo del modelo guardado correctamente.")

### Paso 3.4 — Predecir sobre observaciones nuevas

In [ ]:
observaciones_nuevas = np.array([
    [0.5, 0.3, 0.1],
    [-1.2, -0.8, -0.4],
])

probabilidades = modelo_lluvia.predict(observaciones_nuevas, verbose=0)
predicciones = (probabilidades > 0.5).astype(int)

for i, (prob, pred) in enumerate(zip(probabilidades, predicciones)):
    etiqueta = "lluvioso" if pred[0] == 1 else "no lluvioso"
    print(f"Observación {i+1}: probabilidad={prob[0]:.3f} -> predicción: día {etiqueta}")

In [ ]:
# ✅ Verificación automática
assert probabilidades.shape == (2, 1), "Se esperan 2 probabilidades (una por observación)"
assert ((probabilidades >= 0) & (probabilidades <= 1)).all(), "Las probabilidades deben estar en [0, 1]"
print("✅ Verificación 3.4 OK: predicciones generadas con forma y rango correctos.")

## Resumen para el docente

Todas las celdas marcadas con ✅ deben ejecutarse sin `AssertionError`. Si una entrega de estudiante, adaptada a este notebook, falla en alguna verificación, la tabla de errores comunes del solucionario en `.docx` (Sección 2) cubre las causas más frecuentes observadas en semanas anteriores del curso.